# 02. トリガー - 処理をいつ動かすか

`01` では `availableNow` というトリガーを使いました。深く説明しないまま使っていたので、
ここで正面から扱います。

このノートブックで確かめること:

1. サーバーレスでは**使えないトリガーがある**。まずそれを体験する
2. 使える `availableNow` と `once` は何が違うのか
3. では継続的に取り込みたいときはどうするのか

**前提**: `01_auto_loader` を一通り動かしていること。

## 準備

In [9]:
from databricks.connect import DatabricksSession
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [10]:
CATALOG = "tech_survey"

# 01とは別のテーブル・フォルダを使う。実験が混ざらないようにするため
TABLE = f"{CATALOG}.bronze.orders_trigger"
LANDING = f"/Volumes/{CATALOG}/ops/landing/02_triggers"
CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/02_triggers"

## 1. ファイルを10個置く

今回は「1回の取り込みが、何回のバッチに分かれるか」を見たいので、ファイルを複数用意します。

In [11]:
import json
import random
import uuid

spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

# 対象が無い状態で消そうとするとエラーになる。初回実行ではまだフォルダが無いので、その場合は無視する
for path in (LANDING, CHECKPOINT):
    try:
        dbutils.fs.rm(path, True)
    except NotFound:
        pass


# 何度もファイルを置くので、その部分だけまとめておく
def put_orders(name: str, n: int) -> None:
    rows = []
    for _ in range(n):
        rows.append({"order_id": str(uuid.uuid4()), "amount": random.randint(1000, 50000)})

    text = "\n".join(json.dumps(r) for r in rows)
    dbutils.fs.put(f"{LANDING}/{name}.json", text, True)


# 5件ずつ入ったファイルを10個作る
for i in range(10):
    put_orders(f"orders_{i}", 5)

print("置いたファイル数:", len(dbutils.fs.ls(LANDING)))

置いたファイル数: 10


## 2. 使えないトリガーを試す

トリガーには本来こういう種類があります。

| トリガー | 動き |
|---|---|
| `availableNow=True` | 未処理分を処理したら止まる |
| `once=True` | 未処理分を処理したら止まる (`availableNow` との違いは後述) |
| `processingTime="5 seconds"` | 5秒ごとにバッチを動かし続ける。**止まらない** |
| 指定しない | 前のバッチが終わり次第すぐ次を動かす。**止まらない** |

下2つは「止まらない」トリガーです。常時データが流れてくるシステムではこちらを使います。

ところが、**サーバーレスコンピュートではこの2つが使えません**。
実際にやってみて、どう断られるかを見ておきます。

In [ ]:
# Auto Loaderでストリームを作る
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .load(LANDING)
)

# エラーメッセージを読みたいので、ここでは例外を捕まえて表示する
try:
    df.writeStream.option("checkpointLocation", CHECKPOINT).trigger(
        processingTime="5 seconds"  # ← サーバレスで非サポートなのがここ
    ).toTable(TABLE)
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:400])

AnalysisException
[INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED] Trigger type ProcessingTime is not supported for this cluster type.
Use a different trigger type e.g. AvailableNow, Once. SQLSTATE: 0A000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.infiniteStreamingTriggerNotSupportedError(QueryCompilationErrors.scala:7518)
	at org.apache.spark.sql.strea


`INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED` と出たはずです。  
「終わらないトリガーは、このコンピュートでは対応していない」という意味で、
代わりに `AvailableNow` か `Once` を使えと書かれています。

Databricks Free Edition はサーバーレス専用の環境なので、この制約は回避できません。
常時起動のストリームを動かしたい場合は、クラシックコンピュートが使えるプランが必要になります。

実務上はこれで困らない場面が多いです。理由は `5.` で扱います。

## 3. `availableNow` で取り込む

ここで新しいオプションを1つ足します。

- `cloudFiles.maxFilesPerTrigger` … 1回のマイクロバッチで読むファイル数の上限

10個のファイルに対して上限を3にすると、1回では読み切れません。
`availableNow` は**未処理分が無くなるまでマイクロバッチを繰り返して**から止まります。
つまり何回かに分かれるはずです。

In [ ]:
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/_schema")
    .option("cloudFiles.maxFilesPerTrigger", 3)  # ← 1回のマイクロバッチで処理するファイル数を制限する
    .load(LANDING)
)

query = (
    df.writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)
query.awaitTermination()

# recentProgress には、動いたマイクロバッチの記録が入っている
print("バッチ数:", query.recentProgress)

バッチ数: 4


In [16]:
# 各バッチが何行読んだかを見る
for p in query.recentProgress:
    print(f"batchId={p.batchId}  numInputRows={p.numInputRows}")

batchId=0  numInputRows=15
batchId=1  numInputRows=15
batchId=2  numInputRows=15
batchId=3  numInputRows=5


In [17]:
spark.table(TABLE).count()

50

## 4. `once` と比べる

同じことを `once=True` でやります。ただし**チェックポイントを分ける必要があります**。

`01` で確かめたとおり、同じチェックポイントを使うと処理済みファイルは飛ばされてしまい、
比較になりません。別の場所を指すことで、もう一度10ファイルを最初から読ませます。

In [18]:
TABLE_ONCE = f"{CATALOG}.bronze.orders_trigger_once"
CHECKPOINT_ONCE = f"{CHECKPOINT}_once"

spark.sql(f"DROP TABLE IF EXISTS {TABLE_ONCE}")
try:
    dbutils.fs.rm(CHECKPOINT_ONCE, True)
except NotFound:
    pass

In [22]:
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_ONCE}/_schema")
    .option("cloudFiles.maxFilesPerTrigger", 3)  # 3章と同じ上限をかけている
    .load(LANDING)
)

query_once = (
    df.writeStream.option("checkpointLocation", CHECKPOINT_ONCE)
    .trigger(once=True)
    .toTable(TABLE_ONCE)
)
query_once.awaitTermination()

print("バッチ数:", len(query_once.recentProgress))
for p in query_once.recentProgress:
    print(f"batchId={p.batchId}  numInputRows={p.numInputRows}")

バッチ数: 1
batchId=3  numInputRows=0


In [23]:
spark.table(TABLE_ONCE).count()

50

### 何が違ったか

同じ `maxFilesPerTrigger=3` を指定したのに、バッチの分かれ方が違ったはずです。

- `availableNow` … 上限を守りながら、未処理分が無くなるまでバッチを繰り返す
- `once` … **上限を無視して**、あるものを1回のバッチで全部処理して終わる

最終的な件数は同じでも、途中の動きが違います。
ファイルが大量に溜まっている状況で `once` を使うと、1回のバッチが巨大になり、
メモリ不足で落ちることがあります。`availableNow` なら小分けに処理されるので、その心配が減ります。

こうした理由から `once` は非推奨になっていて、**新しく書くなら `availableNow`** です。
ここで扱ったのは、古いコードで見かけたときに違いが分かるようにするためです。

## 5. 継続的に取り込みたい場合はどうするか

「止まらないトリガーが使えない」となると、常に流れてくるデータをどう扱うのかが問題になります。

答えは **`availableNow` を繰り返し実行する** です。
15分ごとに動かせば、遅れは最大15分。1時間ごとなら最大1時間。求める鮮度に合わせて間隔を決めます。

この繰り返しを担当するのが Databricks の Jobs で、`13_workflows_jobs` で扱います。

常時起動と比べた利点もあります。

- 処理していない間はコンピュートが動かないので安い
- 失敗したときに、次の実行が勝手に続きから拾ってくれる (チェックポイントのおかげ)

逆に秒単位の鮮度が要る場合は、この方式では足りません。そのときはクラシックコンピュートで
常時起動のストリームを動かすことになります。

## 考えてみる

- `3.` のバッチ数は、ファイル10個・上限3で計算した予想と合っていましたか
- `4.` で最終的な件数が `3.` と同じになったのはなぜでしょうか
- 1時間ごとに `availableNow` を動かす設計にしたとき、途中で1回失敗したらどうなるでしょうか

### 答え

**Q1. バッチ数は予想と合っていたか**

4バッチ (3 + 3 + 3 + 1) になります。`maxFilesPerTrigger` は「1回のマイクロバッチで読むファイル数の上限」なので、
10ファイルを3ずつ読むと、10 ÷ 3 を切り上げた4回が必要になります。

**Q2. なぜ最終的な件数が同じになるのか**

トリガーが決めるのは「何回に分けて処理するか」であって、「何を読むか」ではないからです。
`availableNow` も `once` も、対象は同じ10ファイル (50件) です。
どちらも新しいチェックポイントから始めたので、全ファイルを最初から読みました。違うのは途中の刻み方だけです。

**Q3. 1時間ごとの実行が1回失敗したらどうなるか**

次の実行が、失敗した分と新しく増えた分をまとめて処理します。取りこぼしも重複も起きません。

チェックポイントには「どこまで完了したか」が記録されています。`01` で見た `offsets` と `commits` の
二段構えがここで効いていて、完了していないバッチは未完了のまま残るため、次の実行がやり直してくれます。

影響は遅延だけです。1時間以内に届くはずだったデータが、最大2時間かかることになります。
ただし失敗に気づかず放置すると未処理分が溜まり続けるので、失敗通知の設定は別途必要です。

## 後片付け

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
# spark.sql(f"DROP TABLE IF EXISTS {TABLE_ONCE}")

# for path in (LANDING, CHECKPOINT, CHECKPOINT_ONCE):
#     try:
#         dbutils.fs.rm(path, True)
#     except NotFound:
#         pass